# 9. Footprint algebra and pairs

Relations built from other relations, continuing "8. Chains and partial relations" on the same mesh: composing and transposing relations (the footprint algebra), relations declared from such expressions, as ordered stencils or with a `reach`, and sparse fields on the pairs of a relation.

The example mesh, as in 8: a triangle split in four, the interior cell `c0` and the boundary cells `c1`, `c2`, `c3`, with 9 edges and 6 vertices.

```
              v2
             /  \
           e7    e6
           / c3   \
         v5 --e1-- v4
         / \  c0  / \
       e8  e2   e0   e5
       / c1  \  /  c2 \
     v0 --e3-- v3 --e4-- v1
```

Contents: the mesh · footprint algebra · stencils, reach and derived tables · pairs · summary.

## The mesh

The tables of `C2E`, `C2V` and the partial `E2C` (see "8. Chains and partial relations"), with the extents of the dimensions: a transpose, e.g. the cells of each vertex, needs the extent of its target, here `Vertex: 6`, since a vertex may be in no cell.

In [1]:
from concepts import (
    MISSING_VALUE,
    Boundary,
    Dims,
    Field,
    Flatten,
    Location,
    Mesh,
    Neighbor,
    Reduction,
    Relation,
    Unflatten,
)
from implementation import NumPyArrayField, neighbor_table, sum_over


class Cell(Location): ...


class Edge(Location): ...


class Vertex(Location): ...


class C2E(Relation[Cell, Edge], max_neighbors=3): ...


class C2V(Relation[Cell, Vertex], max_neighbors=3): ...


class E2C(Relation[Edge, Cell], max_neighbors=2, partial=True): ...  # boundary edges: 1 cell


M = MISSING_VALUE
tables = {
    C2E: neighbor_table(C2E, [[0, 1, 2], [3, 2, 8], [4, 5, 0], [1, 6, 7]]),
    C2V: neighbor_table(C2V, [[3, 4, 5], [0, 3, 5], [3, 1, 4], [5, 4, 2]]),
    E2C: neighbor_table(
        E2C, [[0, 2], [0, 3], [0, 1], [1, M], [2, M], [2, M], [3, M], [3, M], [1, M]]
    ),
}
extents = {Cell: 4, Edge: 9, Vertex: 6}
split = Mesh(tables, extents=extents)
print("E2C =", split[E2C])

t = NumPyArrayField({Cell: range(4)}, [20.0, 10.0, 30.0, 40.0])  # cell temperatures
outside = 15.0  # the temperature beyond the boundary


def format_step(step):
    if isinstance(step, type):  # a relation
        return step.__name__
    if isinstance(step, Boundary):
        return f"Boundary({step.relation.__name__})"
    if isinstance(step, Reduction):
        return f"Reduction({step.axis.__name__})"
    if isinstance(step, (Flatten, Unflatten)):
        return f"{type(step).__name__}({step.relation.__name__})"
    return f"{step[0].__name__}{step[1]:+d}"


def format_reads(reads):
    return ["(" + ", ".join(map(format_step, read)) + ")" for read in reads]

E2C = Field(Edge[0:9], Neighbor[E2C][0:2]: [[0, 2], [0, 3], [0, 1], [1, -2], [2, -2], [2, -2], [3, -2], [3, -2], [1, -2]])


## Footprint algebra

| Expression | Meaning |
|---|---|
| `R1 @ R2` | The composition: `R1`, then `R2` (`R1.target` must be `R2.source`). Of two relations, a derived relation, anonymous and interned (the same class each time), of width `w1 * w2`: one neighbor position per *path*, so `t(C2E @ E2C)` has one neighbor dimension, `Neighbor[C2E @ E2C]`, where the chained `t(E2C)(C2E)` has two. Partial or with skip values if `R1` or `R2` is; a missing neighbor or skip position of `R1` gives `w2` of them |
| `R.T` | The transpose: the sources of `R` reaching each of its targets. Its width depends on the mesh |
| `Compose`, `Transpose` | The staged expressions with a transpose, e.g. `C2V @ C2V.T`: only used in the declarations of relations (below), not to read fields (a `TypeError`) |

`compose(R1, R2)` is `R1 @ R2`; `composition_of(expr)` gives an expression as a `Compose` (also a composed relation), or `None`, and `expression_name(expr)` its name. Expressions are values, not types: annotations use declared relations. Footprints are sets: `neighbors(R, sources)` gives the neighbors of some sources, without order or multiplicity.

In [2]:
from implementation import neighbors

print(
    (C2E @ E2C).__name__, (C2E @ E2C) is (C2E @ E2C), (C2E @ E2C).max_neighbors, (C2E @ E2C).partial
)
print(C2V.T, "|", C2V @ C2V.T, type(C2V @ C2V.T).__name__)
with split:
    # each edge of a cell, then both its cells: c1, e3 -> [1, M], e2 -> [0, 1], e8 -> [1, M]
    print("C2E @ E2C =", split[C2E @ E2C].data.tolist())
    print("t(C2E @ E2C) =", t(C2E @ E2C))  # only c0 has all its paths
    one_dim = sum_over(t((C2E @ E2C).boundary(outside)), axis=Neighbor[C2E @ E2C])
    chained = t(E2C.boundary(outside))(C2E)
    two_dims = sum_over(sum_over(chained, axis=Neighbor[E2C, Neighbor[C2E]]), axis=Neighbor[C2E])
    print("one neighbor dimension  =", one_dim)  # the heat across the edges of 8: the same values
    print("two neighbor dimensions =", two_dims)
    try:
        t(C2V.T)
    except TypeError as error:
        print("TypeError:", error)
    print("footprint of c1:", neighbors(C2E @ E2C, (1,), boundary=True))  # c0, c1
    try:
        neighbors(C2E @ E2C, (1,))
    except ValueError as error:
        print("ValueError:", error)

C2E @ E2C True 6 True
C2V.T | C2V @ C2V.T Compose
C2E @ E2C = [[0, 2, 0, 3, 0, 1], [1, -2, 0, 1, 1, -2], [2, -2, 2, -2, 0, 2], [0, 3, 3, -2, 3, -2]]
t(C2E @ E2C) = Field(Cell[0:1], Neighbor[C2E @ E2C][0:6]: [[20.0, 30.0, 20.0, 40.0, 20.0, 10.0]])
one neighbor dimension  = Field(Cell[0:4]: [140.0, 80.0, 140.0, 170.0])
two neighbor dimensions = Field(Cell[0:4]: [140.0, 80.0, 140.0, 170.0])
TypeError: `C2V.T` has no table of a fixed width: declare a relation with it, `stencil=Stencil(C2V.T)`, and premap by that relation.
footprint of c1: range(0, 2)
ValueError: Cell[1:2] is beyond the computable domain: it has missing neighbors through 'C2E @ E2C' (read with `(C2E @ E2C).boundary(value)` there).


## Stencils, reach and derived tables

A declared relation may carry its construction from an expression:

- `stencil=Stencil(expr, paths=..., include_self=...)`: an ordered table, derived by the mesh. `paths="all"` keeps all the paths; `"exclude_self"` drops the source from the targets of each first hop, keeping the order and the missing neighbors (ICON's `C2E2C`, from `C2E @ E2C`); `"unique"` keeps each neighbor once, in the order of first occurrence. `include_self=True` puts the source first (ICON's `C2E2CO`). Rows are padded with `SKIP_VALUE` to `max_neighbors`.
- `reach=expr`: for tables in an order that cannot be derived, e.g. ICON's special orders: each row of the given table must be within the footprint of `expr`.

A given table of a relation with a stencil must be the derived one, and the rows of one with a `reach` within its footprint: else a `ValueError`. The derived tables are derived on first use and cached per mesh; they are not keys of the mesh.

In [3]:
from concepts import Stencil


class C2E2C(
    Relation[Cell, Cell],
    max_neighbors=3,
    partial=True,
    stencil=Stencil(C2E @ E2C, paths="exclude_self"),
): ...


class C2E2CO(
    Relation[Cell, Cell],
    max_neighbors=4,
    partial=True,
    stencil=Stencil(C2E @ E2C, paths="exclude_self", include_self=True),
): ...


class V2C(Relation[Vertex, Cell], max_neighbors=3, skip_values=True, stencil=Stencil(C2V.T)): ...


class C2V2C(
    Relation[Cell, Cell], max_neighbors=4, stencil=Stencil(C2V @ C2V.T, paths="unique")
): ...


class C2E2Cr(Relation[Cell, Cell], max_neighbors=3, partial=True, reach=C2E @ E2C): ...


given = neighbor_table(C2E2Cr, [[1, 2, 3], [0, M, M], [0, M, M], [0, M, M]])  # an order of its own
split = Mesh({**tables, C2E2Cr: given}, extents=extents)
for relation in (C2E2C, C2E2CO, V2C, C2V2C, C2E2Cr):
    print(f"{relation.__name__:6}", split[relation].data.tolist())
print("cached:", split[C2E2C] is split[C2E2C], "| keys:", [r.__name__ for r in split])

wrong = {
    C2E2C: [[1, 2, 3], [M, 0, M], [M, M, 0], [0, M, M]],  # c0: sorted, not in the order of C2E
    C2E2Cr: [[1, 2, 3], [2, M, M], [0, M, M], [0, M, M]],  # c2 is not across an edge of c1
}
for relation, rows in wrong.items():
    try:
        Mesh({**tables, relation: neighbor_table(relation, rows)}, extents=extents)
    except ValueError as error:
        print("ValueError:", error)
try:
    Mesh(tables)[V2C]  # without the extent of Vertex
except ValueError as error:
    print("ValueError:", error)

C2E2C  [[2, 3, 1], [-2, 0, -2], [-2, -2, 0], [0, -2, -2]]
C2E2CO [[0, 2, 3, 1], [1, -2, 0, -2], [2, -2, -2, 0], [3, 0, -2, -2]]
V2C    [[1, -1, -1], [2, -1, -1], [3, -1, -1], [0, 1, 2], [0, 2, 3], [0, 1, 3]]
C2V2C  [[0, 1, 2, 3], [1, 0, 2, 3], [0, 1, 2, 3], [0, 1, 3, 2]]
C2E2Cr [[1, 2, 3], [0, -2, -2], [0, -2, -2], [0, -2, -2]]
cached: True | keys: ['C2E', 'C2V', 'E2C', 'C2E2Cr']
ValueError: The table of 'C2E2C' is not the one of its stencil: the row of 0 is [1, 2, 3], not [2, 3, 1].
ValueError: The table of 'C2E2Cr' has the row [2, -2, -2] for 1: its neighbors must be distinct, within its reach, `C2E @ E2C`, [-2, 0, 1], missing only if it is.
ValueError: `C2V.T` needs the extent of 'Vertex': give it to the mesh, `extents={Vertex: ...}`.


In [4]:
import contextlib
import io

from implementation import column_major, compiled, eager, field_operator, lazy, lazy_backward


@field_operator()
def blend(t: Field[Dims[Cell], float], outside: float) -> Field[Dims[Cell], float]:
    return 0.25 * (t + sum_over(t(C2E2C.boundary(outside)), axis=Neighbor[C2E2C]))


with split:
    # c0: (20 + 30 + 40 + 10) / 4 = 25; c1: (10 + 15 + 20 + 15) / 4 = 15
    print("blend =", blend(t, outside))
    for backend in (eager, lazy, lazy_backward, column_major, compiled):
        with contextlib.redirect_stdout(io.StringIO()):  # silence the backends
            values = backend(blend)[{Cell: (0, 3)}](t, outside).data.tolist()
        print(f"{backend}: {values}")
    for backend in (lazy, lazy_backward):  # twice, for c1: (15 + 15 + 25 + 15) / 4 = 17.5
        print(f"--- {backend}")
        op = backend(blend)
        print(op[{Cell: (1,)}](op(t, outside), outside))  # lazy_backward: c1 needs c0, c1

blend = Field(Cell[0:4]: [25.0, 15.0, 20.0, 22.5])
eager: [25.0, 22.5]
lazy: [25.0, 22.5]
lazy_backward: [25.0, 22.5]
column_major: [25.0, 22.5]
compiled: [25.0, 22.5]
--- lazy
[lazy] computed Cell[0:4], kept Cell[0:4]
[lazy] computed Cell[0:4], kept Cell{1}
Field(Cell{1}: [17.5])
--- lazy_backward
[lazy_backward] computed Cell{1}
[lazy_backward] computed Cell[0:2]
Field(Cell{1}: [17.5])


## Pairs

The neighbors of a relation flatten to its *pairs*: a sparse field, like COO, without padding.

| Concept | Kind | Meaning |
|---|---|---|
| `Pairs[R]` | dimension | A `Location` with one point per neighbor in the table of `R` (not per skip position), in row-major order. One per relation; a partial relation has none (a `TypeError`): its missing neighbors have no target |
| `R.to_pairs` | relation | From `R.source` to `Pairs[R]`: the pair of each neighbor (`SKIP_VALUE` at the skip positions), derived by the mesh |
| `R.pair_source`, `R.pair_target` | relations | *Functional*: from each pair to its source, and to its target. A gather through them replaces the root, without a neighbor dimension: `t(V2C.pair_target)` has the dimension `Pairs[V2C]`. Only on fields without neighbor dimensions |
| `flatten(f, axis=Neighbor[R])` | DSL builtin | `(R.source, Neighbor[R], ...)` to `(Pairs[R], ...)`: read step `Flatten(R)` |
| `unflatten(p, axis=Pairs[R])` | DSL builtin | Back, with 0 at the skip positions: read step `Unflatten(R)` |

`pairs_of(R)` is `Pairs[R]`, and `pair_relation(R, role)` is `R.to_pairs`, `R.pair_source` or `R.pair_target`. A pair is a neighbor position, a path, not a distinct neighbor: `Pairs[C2E @ E2C]` has a point per path, even if paths have the same ends.

In [5]:
from concepts import Pairs
from implementation import flatten, unflatten

print(Pairs[V2C].__name__, "| a Location:", issubclass(Pairs[V2C], Location))
with split:
    print("V2C.to_pairs    =", split[V2C.to_pairs].data.tolist())
    for relation in (V2C.pair_source, V2C.pair_target):  # functional: one neighbor
        print(f"{relation.__name__:15} =", split[relation].data[:, 0].tolist())
    flat = flatten(t(V2C), axis=Neighbor[V2C])  # the temperatures of the cells of each vertex
    print("flatten   =", flat)
    print("by target =", t(V2C.pair_target))  # the same, gathered pair by pair
    print("unflatten =", unflatten(flat, axis=Pairs[V2C]))  # padded with 0

Pairs[V2C] | a Location: True
V2C.to_pairs    = [[0, -1, -1], [1, -1, -1], [2, -1, -1], [3, 4, 5], [6, 7, 8], [9, 10, 11]]
V2C.pair_source = [0, 1, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5]
V2C.pair_target = [1, 2, 3, 0, 1, 2, 0, 2, 3, 0, 1, 3]
flatten   = Field(Pairs[V2C][0:12]: [10.0, 30.0, 40.0, 20.0, 10.0, 30.0, 20.0, 30.0, 40.0, 20.0, 10.0, 40.0])
by target = Field(Pairs[V2C][0:12]: [10.0, 30.0, 40.0, 20.0, 10.0, 30.0, 20.0, 30.0, 40.0, 20.0, 10.0, 40.0])
unflatten = Field(Vertex[0:6], Neighbor[V2C][0:3]: [[10.0, 0.0, 0.0], [30.0, 0.0, 0.0], [40.0, 0.0, 0.0], [20.0, 10.0, 30.0], [20.0, 30.0, 40.0], [20.0, 10.0, 40.0]])


In [6]:
from implementation import backward_domain, format_domain

# interpolation weights on the pairs: 1 at a vertex of one cell; else 0.5 for c0, 0.25 for the others
w = NumPyArrayField(
    {Pairs[V2C]: range(12)}, [1.0, 1.0, 1.0, 0.5, 0.25, 0.25, 0.5, 0.25, 0.25, 0.5, 0.25, 0.25]
)


@field_operator()
def to_vertices(
    t: Field[Dims[Cell], float], w: Field[Dims[Pairs[V2C]], float]
) -> Field[Dims[Vertex], float]:
    weighted = w * t(V2C.pair_target)  # on Pairs[V2C]
    return sum_over(unflatten(weighted, axis=Pairs[V2C]), axis=Neighbor[V2C])


with split:
    # v3: 0.5 * 20 + 0.25 * 10 + 0.25 * 30 = 20; v4: 0.5 * 20 + 0.25 * 30 + 0.25 * 40 = 27.5
    print("to_vertices =", to_vertices(t, w))
    for backend in (eager, lazy, lazy_backward, column_major, compiled):
        with contextlib.redirect_stdout(io.StringIO()):  # silence the backends
            values = backend(to_vertices)[{Vertex: (3, 4)}](t, w).data.tolist()
        print(f"{backend}: {values}")
    for name, reads in to_vertices.__gt_extents__.items():
        needed = format_domain(backward_domain({Vertex: (3,)}, reads))
        print(f"{name}:", *format_reads(reads), "| needed for Vertex{3}:", needed)
    # 9 + 7 + 7 + 7 paths, through 4 distinct cells per row: one pair per path
    print(
        "pairs of C2V @ V2C:", format_domain(flatten(t(C2V @ V2C), axis=Neighbor[C2V @ V2C]).domain)
    )
try:
    Pairs[E2C]
except TypeError as error:
    print("TypeError:", error)

to_vertices = Field(Vertex[0:6]: [10.0, 30.0, 40.0, 20.0, 27.5, 22.5])
eager: [20.0, 27.5]
lazy: [20.0, 27.5]
lazy_backward: [20.0, 27.5]
column_major: [20.0, 27.5]
compiled: [20.0, 27.5]
t: (V2C.pair_target, Unflatten(V2C), Reduction(Neighbor[V2C])) | needed for Vertex{3}: Cell[0:3]
w: (Unflatten(V2C), Reduction(Neighbor[V2C])) | needed for Vertex{3}: Pairs[V2C][3:6]
pairs of C2V @ V2C: Pairs[C2V @ V2C][0:30]
TypeError: 'E2C' is partial: it has no pairs, since its missing neighbors have no target.


## Summary

| Feature | What is checked |
|---|---|
| `R1 @ R2`, `R.T` | the targets and sources match (a `TypeError`); expressions with a transpose only in declarations (a `TypeError`) |
| `C2E @ E2C` | one neighbor position per path; partial and skip values propagate; backward, missing paths without a boundary value (a `ValueError`) |
| `stencil=Stencil(...)` | the table derived on first use, cached per mesh; a given table equal to the derived one (a `ValueError`) |
| `reach=expr` | the rows of the given table distinct and within the footprint of `expr` (a `ValueError`) |
| `Stencil(R.T)` | the extent of the target known to the mesh (a `ValueError`) |
| `Pairs[R]`, `flatten`, `unflatten` | no pairs for partial relations (a `TypeError`); functional gathers only on fields without neighbor dimensions (a `NotImplementedError`) |